# Querying Pandas DataFrames

This notebook focuses on extracting subsets of data using **Boolean Masking**. A Boolean Mask is an array of True/False values that you can apply to a DataFrame to filter data.

Topics covered:
- Creating Boolean masks
- Using the `.where()` function and `.dropna()`
- Direct bracket filtering `df[mask]`
- Combining multiple logical conditions (`&`, `|`)
- Using Pandas logical functions (`.gt()`, `.lt()`)

In [1]:
# Import pandas library
import pandas as pd

## Loading the Dataset

We use the Graduate Admissions dataset and clean its columns.

In [2]:
# Load data, set index, and strip whitespace from column titles
df = pd.read_csv("datasets/Admission_Predict.csv", index_col=0)
df.columns = [x.upper().strip() for x in df.columns]

df.head()

,GRE SCORE,TOEFL SCORE,UNIVERSITY RATING,SOP,LOR,CGPA,RESEARCH,CHANCE OF ADMIT
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
3,316,104,3,3.0,3.5,8.00,1,0.72
4,322,110,3,3.5,2.5,8.67,1,0.80
5,314,103,2,2.0,3.0,8.21,0,0.65


## Generating a Boolean Mask

A boolean mask is generated by evaluating a condition against a column. Let's find students with a chance of admission greater than 70%.

In [3]:
# Compare a Series to a scalar. Returns a Series of booleans.
admitMask = df['CHANCE OF ADMIT'] > 0.7
admitMask

Serial No.
1       True
2       True
3       True
4       True
5      False
       ...  
396     True
397     True
398     True
399    False
400     True
Name: CHANCE OF ADMIT, Length: 400, dtype: bool

## Applying a Mask using `.where()`

The `.where()` method applies the boolean mask to the DataFrame. If the mask is `True` for a row, it keeps the data. If it's `False`, it fills the entire row with `NaN` (Not a Number).

In [4]:
# Apply the mask. Notice the NaN rows where the condition failed.
df.where(admitMask).head()

,GRE SCORE,TOEFL SCORE,UNIVERSITY RATING,SOP,LOR,CGPA,RESEARCH,CHANCE OF ADMIT
Serial No.,,,,,,,,
1,337.0,118.0,4.0,4.5,4.5,9.65,1.0,0.92
2,324.0,107.0,4.0,4.0,4.5,8.87,1.0,0.76
3,316.0,104.0,3.0,3.0,3.5,8.00,1.0,0.72
4,322.0,110.0,3.0,3.5,2.5,8.67,1.0,0.80
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Cleaning up after `.where()`

Usually, we don't want rows full of `NaN`. We can chain the `.dropna()` method to instantly discard any row containing missing data.

In [5]:
# Drop rows that were replaced with NaNs by the where function
df.where(admitMask).dropna().head()

,GRE SCORE,TOEFL SCORE,UNIVERSITY RATING,SOP,LOR,CGPA,RESEARCH,CHANCE OF ADMIT
Serial No.,,,,,,,,
1,337.0,118.0,4.0,4.5,4.5,9.65,1.0,0.92
2,324.0,107.0,4.0,4.0,4.5,8.87,1.0,0.76
3,316.0,104.0,3.0,3.0,3.5,8.00,1.0,0.72
4,322.0,110.0,3.0,3.5,2.5,8.67,1.0,0.80
6,330.0,115.0,5.0,4.5,3.0,9.34,1.0,0.90


## Querying Shortcut: The Overloaded Bracket Operator

Pandas developers overloaded the `[]` operator so you can simply place the boolean mask inside it. This automatically applies `.where()` and `.dropna()` simultaneously!

In [6]:
# This is the standard, most common way to filter data in Pandas.
df[df['CHANCE OF ADMIT'] > 0.7].head()

,GRE SCORE,TOEFL SCORE,UNIVERSITY RATING,SOP,LOR,CGPA,RESEARCH,CHANCE OF ADMIT
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
3,316,104,3,3.0,3.5,8.00,1,0.72
4,322,110,3,3.5,2.5,8.67,1,0.80
6,330,115,5,4.5,3.0,9.34,1,0.90


In [7]:
# Note: Standard string indexing still extracts the column
df['GRE SCORE'].head()

Serial No.
1    337
2    324
3    316
4    322
5    314
Name: GRE SCORE, dtype: int64

In [8]:
# Passing a list of strings extracts multiple columns into a new DataFrame
df[['GRE SCORE', "TOEFL SCORE"]].head()

,GRE SCORE,TOEFL SCORE
Serial No.,,
1,337,118
2,324,107
3,316,104
4,322,110
5,314,103


In [9]:
# Combining column extraction with condition filtering
# e.g. "Give me the data of students whose GRE score is > 320"
df[df["GRE SCORE"] > 320].head()

,GRE SCORE,TOEFL SCORE,UNIVERSITY RATING,SOP,LOR,CGPA,RESEARCH,CHANCE OF ADMIT
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
4,322,110,3,3.5,2.5,8.67,1,0.80
6,330,115,5,4.5,3.0,9.34,1,0.90
7,321,109,3,3.0,4.0,8.20,1,0.75


## Combining Multiple Boolean Conditions

When filtering with multiple conditions, you must use bitwise operators like `&` (AND) and `|` (OR) instead of the Python keywords `and`/`or`.

**Important:** Due to operator precedence in Python, each condition MUST be enclosed in parentheses `()`.

In [10]:
# Finding students whose chance of admit is between 0.7 and 0.9
(df["CHANCE OF ADMIT"] > 0.7) & (df["CHANCE OF ADMIT"] < 0.9)

Serial No.
1      False
2       True
3       True
4       True
5      False
       ...  
396     True
397     True
398    False
399    False
400    False
Name: CHANCE OF ADMIT, Length: 400, dtype: bool

### Using Built-in Pandas Logical Functions

An alternative to bitwise operators with parentheses is utilizing built-in functions like `.gt()` (greater than), `.lt()` (less than), `.ge()` (greater/equal), and `.le()` (less/equal).

In [11]:
# This produces the exact same mask as the bitwise operator method above
df["CHANCE OF ADMIT"].gt(0.7) & df["CHANCE OF ADMIT"].lt(0.9)

Serial No.
1      False
2       True
3       True
4       True
5      False
       ...  
396     True
397     True
398    False
399    False
400    False
Name: CHANCE OF ADMIT, Length: 400, dtype: bool

In [12]:
# These methods can be chained cleanly to avoid the '&' operator entirely
df["CHANCE OF ADMIT"].gt(0.7).lt(0.9)

Serial No.
1      False
2      False
3      False
4      False
5       True
       ...  
396    False
397    False
398    False
399     True
400    False
Name: CHANCE OF ADMIT, Length: 400, dtype: bool